**Problem 8**

The state wildlife biologists want to model how many fish are being caught by fishermen at a state park. Visitors are asked how long they stayed, how many people were in the group, whether there were children in the group, and how many fish were caught. Some visitors do not fish, but there is no data on whether a person fished or not. Some visitors who did fish did not catch any fish, so there are excess zeros in the data because of the people that did not fish. The dataset is taken from:

UCLA Fish Dataset

Fit the Zero-Inflated Poisson (ZIP) regression generalized linear model (GLM) to identify the factors associated with the number of fish caught. Interpret the results.

In [1]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.discrete.count_model import ZeroInflatedPoisson

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

In [2]:
url = "https://stats.idre.ucla.edu/stat/data/fish.csv"

df = pd.read_csv(url)

In [3]:
df.head()

,nofish,livebait,camper,persons,child,xb,zg,count
0,1,0,0,1,0,-0.896315,3.050405,0
1,0,1,1,1,0,-0.558345,1.746149,0
2,0,1,0,1,0,-0.401731,0.279939,0
3,0,1,1,2,1,-0.956298,-0.601526,0
4,0,1,0,1,0,0.436891,0.527709,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   nofish    250 non-null    int64  
 1   livebait  250 non-null    int64  
 2   camper    250 non-null    int64  
 3   persons   250 non-null    int64  
 4   child     250 non-null    int64  
 5   xb        250 non-null    float64
 6   zg        250 non-null    float64
 7   count     250 non-null    int64  
dtypes: float64(2), int64(6)
memory usage: 15.8 KB


In [5]:
df.describe()

,nofish,livebait,camper,persons,child,xb,zg,count
count,250.000000,250.000000,250.000000,250.00000,250.000000,250.000000,250.000000,250.000000
mean,0.296000,0.864000,0.588000,2.52800,0.684000,0.973796,0.252323,3.296000
std,0.457407,0.343476,0.493182,1.11273,0.850315,1.440277,2.102391,11.635028
min,0.000000,0.000000,0.000000,1.00000,0.000000,-3.275050,-5.625944,0.000000
25%,0.000000,1.000000,0.000000,2.00000,0.000000,0.008267,-1.252724,0.000000
50%,0.000000,1.000000,1.000000,2.00000,0.000000,0.954550,0.605079,0.000000
75%,1.000000,1.000000,1.000000,4.00000,1.000000,1.963855,1.993237,2.000000
max,1.000000,1.000000,1.000000,4.00000,3.000000,5.352674,4.263185,149.000000


In [6]:
#Check Excess Zeros
zeros = (df['count'] == 0).sum()

print("Number of Zeros:", zeros)

print(
    "Percentage of Zeros:",
    zeros / len(df) * 100
)

Number of Zeros: 142
Percentage of Zeros: 56.8


In [7]:
#Prepare Variables

#Response variable:

y = df['count']

In [8]:
#Predictor variables:

X = df[['child', 'persons', 'camper']]

In [9]:
#Add constant:

X = sm.add_constant(X)

In [10]:
#Fit Zero-Inflated Poisson Model

#Model: count∼child+persons+camper

zip_model = ZeroInflatedPoisson(
    endog=y,
    exog=X,
    exog_infl=X,
    inflation='logit'
).fit()

Optimization terminated successfully.
         Current function value: 3.010926
         Iterations: 29
         Function evaluations: 31
         Gradient evaluations: 31


In [11]:
print(zip_model.summary())

                     ZeroInflatedPoisson Regression Results                    
Dep. Variable:                   count   No. Observations:                  250
Model:             ZeroInflatedPoisson   Df Residuals:                      246
Method:                            MLE   Df Model:                            3
Date:                 Sun, 10 May 2026   Pseudo R-squ.:                  0.3321
Time:                         12:14:42   Log-Likelihood:                -752.73
converged:                        True   LL-Null:                       -1127.0
Covariance Type:             nonrobust   LLR p-value:                6.123e-162
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
inflate_const       1.6636      0.503      3.308      0.001       0.678       2.649
inflate_child       1.9046      0.322      5.915      0.000       1.273       2.536
inflate_persons    -0.92

**Interpretation of Zero-Inflated Poisson (ZIP) Regression Results**

The Zero-Inflated Poisson model was used because the dataset contains excess zeros, meaning many visitors caught no fish, possibly because some individuals did not fish at all.

The overall model is highly significant:

Likelihood Ratio p-value = 6.12 × 10⁻¹⁶²
Pseudo R² = 0.332

This indicates that the predictors explain a substantial portion of the variation in fish counts.

**Count Model Interpretation**

These coefficients explain factors affecting the number of fish caught.

Number of Children (child)
Coefficient = -1.1367
p-value < 0.001

This variable is statistically significant.

Groups with more children are expected to catch fewer fish.

Number of Persons (persons)
Coefficient = 0.8290
p-value < 0.001

This variable is statistically significant.

Larger groups are associated with higher expected fish counts.

**Camper Status (camper)**

Coefficient = 0.7243
p-value < 0.001

Campers are expected to catch more fish compared to non-campers.

**Inflation Model Interpretation**

The inflation part models the probability of excess zeros.

**inflate_child**

**Positive coefficient:**

More children increase the probability of belonging to the “always zero” group.

Meaning:groups with more children are more likely not to fish at all.
inflate_persons

**Negative coefficient:**

Larger groups are less likely to belong to the excess-zero group.

Meaning:larger groups are more likely to participate in fishing.
inflate_camper

**Negative coefficient:**

Campers are less likely to belong to the excess-zero group.

Meaning:campers are more likely to actually fish.


**Overall Conclusion**


The ZIP regression model indicates that:

larger groups and campers tend to catch more fish,
groups with more children tend to catch fewer fish,
and the excess zeros are partly explained by visitors who likely did not participate in fishing activities.